# Criar as tabelas no SQLite

In [ ]:
import os
import sqlite3
import pandas as pd
from sqlalchemy import create_engine
# Caminho para a pasta Data e subpastas
data_path = r"H:\Projeto USE\data_analytics_jr_test\Data\Data\Data"
subfolders = ["Escolas", "Perfil dos educandos"]
# Conectar ao banco de dados SQLite com SQLAlchemy
engine = create_engine("sqlite:///test_analytics.db")
conn = sqlite3.connect("test_analytics.db")
cursor = conn.cursor()
# Listar arquivos nas subpastas
for subfolder in subfolders:
    folder_path = os.path.join(data_path, subfolder)
    
    if os.path.exists(folder_path):  # Verifica se a subpasta existe
        files = os.listdir(folder_path)
        
        for file in files:
            file_path = os.path.join(folder_path, file)
            table_name = os.path.splitext(file)[0]  # Nome da tabela = nome do arquivo (sem extensão)
            
            if file.endswith(".csv"):
                df = pd.read_csv(file_path, encoding='latin1', delimiter=';')
            elif file.endswith(".xlsx"):
                df = pd.read_excel(file_path)
            else:
                continue  # Pula arquivos que não são CSV ou XLSX
            
            # Escrever no banco de dados
            df.to_sql(table_name, engine, if_exists="replace", index=False)
            print(f"Tabela '{table_name}' criada com sucesso!")
    else:
        print(f"A subpasta '{subfolder}' não foi encontrada.")

# Diagnosticar as colunas das tabelas que contém "escola" especificadas

In [1]:
import sqlite3

# Defina o caminho do seu banco de dados
db_path = r"H:\Projeto USE\data_analytics\db\test_analytics.db"

# Padrão de cabeçalhos
padrão_cabeçalhos = [
    "DRE", "CODESC", "TIPOESC", "NOMESC", "NOMESCOF", "CEU", "DIRETORIA", 
    "SUBPREF", "ENDERECO", "NUMERO", "BAIRRO", "CEP", "TEL1", "TEL2", 
    "FAX", "SITUACAO", "CODDIST", "DISTRITO", "SETOR", "CODINEP", "CODCIE", 
    "EH", "FX_ETARIA", "DT_CRIACAO", "ATO_CRIACAO", "DOM_CRIACAO", "DT_INI_FUNC", 
    "DT_INI_CONV", "DT_AUTORIZA", "DT_EXTINCAO", "NOME_ANT", "REDE", "LATITUDE", 
    "LONGITUDE", "DATABASE"
]

# Função para verificar os cabeçalhos das tabelas
def verificar_cabecalhos():
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()

    # Pegue a lista de tabelas
    cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
    tabelas = cursor.fetchall()

    # Filtrar tabelas que contém 'escola' no nome
    tabelas_escolas = [t[0] for t in tabelas if 'escola' in t[0].lower()]

    # Verificar cabeçalhos de cada tabela
    tabelas_com_problema = {}
    for tabela in tabelas_escolas:
        # Envolver o nome da tabela com aspas invertidas para evitar problemas com caracteres especiais
        cursor.execute(f"PRAGMA table_info(`{tabela}`);")
        colunas = cursor.fetchall()
        nomes_colunas = [col[1] for col in colunas]
        
        # Comparar com o padrão
        colunas_faltando = list(set(padrão_cabeçalhos) - set(nomes_colunas))
        colunas_diferentes = list(set(nomes_colunas) - set(padrão_cabeçalhos))

        if colunas_faltando or colunas_diferentes:
            tabelas_com_problema[tabela] = {
                "colunas_faltando": colunas_faltando,
                "colunas_diferentes": colunas_diferentes
            }

    # Fechar conexão
    conn.close()
    
    return tabelas_com_problema

# Executar e exibir resultados
problemas = verificar_cabecalhos()
if problemas:
    for tabela, info in problemas.items():
        print(f"Tabela: {tabela}")
        
        if info["colunas_faltando"]:
            print("Colunas faltando:")
            for coluna in info["colunas_faltando"]:
                print(f"  - {coluna}")
        
        if info["colunas_diferentes"]:
            print("Colunas com nome diferente:")
            for coluna in info["colunas_diferentes"]:
                print(f"  - {coluna}")
        
        print()
else:
    print("Todas as tabelas com 'escola' no nome estão com os cabeçalhos padrão.")


Tabela: dicionarioescolas
Colunas faltando:
  - DT_INI_FUNC
  - DT_CRIACAO
  - DT_INI_CONV
  - FX_ETARIA
  - BAIRRO
  - NOME_ANT
  - CODCIE
  - NUMERO
  - SUBPREF
  - ATO_CRIACAO
  - SITUACAO
  - LONGITUDE
  - CODESC
  - DISTRITO
  - SETOR
  - DIRETORIA
  - NOMESC
  - DATABASE
  - CEP
  - DRE
  - CODINEP
  - EH
  - ENDERECO
  - FAX
  - DT_EXTINCAO
  - TIPOESC
  - TEL1
  - CEU
  - DT_AUTORIZA
  - REDE
  - LATITUDE
  - NOMESCOF
  - DOM_CRIACAO
  - CODDIST
  - TEL2
Colunas com nome diferente:
  - DESCRIÇÃO
  - CAMPO

Tabela: escolas122023
Colunas com nome diferente:
  - FX_ETARIA.1
  - DESLOC

Tabela: escolasr34
Colunas com nome diferente:
  - DTURNOS13
  - T2D3D15
  - DTURNOS09
  - DTURNOS12
  - T2D3D13
  - T2D3D
  - T2D3D14
  - DTURNOS10
  - DTURNOS
  - T2D3D08
  - DTURNOS11
  - DTURNOS14
  - T2D3D10
  - T2D3D09
  - DTURNOS15
  - DTURNOS08
  - T2D3D12
  - T2D3D11
  - DTURNOS07
  - T2D3D07

Tabela: escolasr34dez2017
Colunas com nome diferente:
  - DTURNOS16
  - DTURNOS13
  - T2D3D15
  - 

# Ajustar as tabelas "escola"

In [10]:
import sqlite3
import difflib
import re

# Função para limpar os nomes das colunas, transformando tudo em minúsculas e removendo caracteres especiais
def limpar_nome_coluna(nome):
    nome = nome.lower()  # Transformar para minúsculas
    nome = re.sub(r'\W+', '', nome)  # Remover caracteres especiais (não alfanuméricos)
    return nome

# Função para corrigir ou adicionar as colunas
def corrigir_ou_adicionar_colunas(tabela, colunas_faltando, colunas_diferentes):
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()

    # Limpar os nomes das colunas faltantes e com nome diferente
    colunas_faltando_limpa = [limpar_nome_coluna(coluna) for coluna in colunas_faltando]
    colunas_diferentes_limpa = [limpar_nome_coluna(coluna) for coluna in colunas_diferentes]

    for coluna_faltando, coluna_faltando_limpa in zip(colunas_faltando, colunas_faltando_limpa):
        # Encontrar coluna com nome semelhante (por exemplo, uma letra a mais ou minúscula/maiúscula)
        colunas_similares = difflib.get_close_matches(coluna_faltando_limpa, colunas_diferentes_limpa, n=1, cutoff=0.8)

        if colunas_similares:
            coluna_errada_limpa = colunas_similares[0]
            coluna_errada = colunas_diferentes[colunas_diferentes_limpa.index(coluna_errada_limpa)]
            # Corrigir o nome da coluna
            cursor.execute(f"ALTER TABLE `{tabela}` RENAME COLUMN `{coluna_errada}` TO `{coluna_faltando}`;")
            print(f"Corrigido nome da coluna '{coluna_errada}' para '{coluna_faltando}' na tabela '{tabela}'")
        else:
            # Verificar se a coluna já existe antes de adicionar
            cursor.execute(f"PRAGMA table_info(`{tabela}`);")
            colunas_existentes = [col[1] for col in cursor.fetchall()]
            
            if coluna_faltando not in colunas_existentes:
                # Se não encontrar, criar a coluna faltante com dados nulos
                cursor.execute(f"ALTER TABLE `{tabela}` ADD COLUMN `{coluna_faltando}` TEXT;")
                print(f"Criada a coluna faltante '{coluna_faltando}' na tabela '{tabela}'")
            else:
                print(f"A coluna '{coluna_faltando}' já existe na tabela '{tabela}'")

    conn.commit()
    conn.close()

# Função para verificar os cabeçalhos e corrigir
def verificar_e_corrigir_cabecalhos():
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()

    # Pegue a lista de tabelas
    cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
    tabelas = cursor.fetchall()

    # Filtrar tabelas que contém 'escola' no nome e não contém 'dicionario'
    tabelas_escolas = [t[0] for t in tabelas if 'escola' in t[0].lower() and 'dicionario' not in t[0].lower()]

    # Verificar cabeçalhos de cada tabela
    for tabela in tabelas_escolas:
        # Envolver o nome da tabela com aspas invertidas para evitar problemas com caracteres especiais
        cursor.execute(f"PRAGMA table_info(`{tabela}`);")
        colunas = cursor.fetchall()
        nomes_colunas = [col[1] for col in colunas]

        # Comparar com o padrão
        colunas_faltando = list(set(padrão_cabeçalhos) - set(nomes_colunas))
        colunas_diferentes = list(set(nomes_colunas) - set(padrão_cabeçalhos))

        if colunas_faltando or colunas_diferentes:
            print(f"Tabela: {tabela}")
            
            # Corrigir ou adicionar as colunas faltantes e com nome diferente
            corrigir_ou_adicionar_colunas(tabela, colunas_faltando, colunas_diferentes)
    
    conn.close()

# Executar e corrigir cabeçalhos
verificar_e_corrigir_cabecalhos()


Tabela: escolas122018
Criada a coluna faltante 'NOMESC' na tabela 'escolas122018'
Tabela: escolas122019
Criada a coluna faltante 'NOMESC' na tabela 'escolas122019'
Tabela: escolas122023
Tabela: escolasr34
Tabela: escolasr34dez2017


# Criar um nova tabela "escolas_full"

In [6]:
import sqlite3

# Caminho do banco de dados
db_path = r"H:\Projeto USE\data_analytics\db\test_analytics.db"

# Função para criar a tabela "escolas_full"
def criar_tabela_escolas_full():
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()

    # Criar a tabela "escolas_full"
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS escolas_full (
            DRE TEXT,
            CODESC BIGINT,
            TIPOESC TEXT,
            NOMESC TEXT,
            NOMESCOF TEXT,
            CEU TEXT,
            DIRETORIA TEXT,
            SUBPREF TEXT,
            ENDERECO TEXT,
            NUMERO TEXT,
            BAIRRO TEXT,
            CEP BIGINT,
            TEL1 TEXT,
            TEL2 TEXT,
            FAX TEXT,
            SITUACAO TEXT,
            CODDIST BIGINT,
            DISTRITO TEXT,
            SETOR FLOAT,
            CODINEP FLOAT,
            CODCIE FLOAT,
            EH FLOAT,
            FX_ETARIA TEXT,
            DT_CRIACAO TEXT,
            ATO_CRIACAO TEXT,
            DOM_CRIACAO TEXT,
            DT_INI_FUNC TEXT,
            DT_INI_CONV TEXT,
            DT_AUTORIZA TEXT,
            DT_EXTINCAO FLOAT,
            NOME_ANT TEXT,
            REDE TEXT,
            LATITUDE BIGINT,
            LONGITUDE BIGINT,
            DATABASE TEXT
        );
    """)

    conn.commit()
    conn.close()

# Criar a tabela "escolas_full"
criar_tabela_escolas_full()

# Inserir os dados na nova tabela criada "escolas_full"

In [7]:
import sqlite3
from datetime import datetime
from dateutil import parser
import re

# Caminho do banco de dados
db_path = r"H:\Projeto USE\data_analytics\db\test_analytics.db"

# Mapeamento de meses abreviados em português para números
meses_abreviados = {
    "jan": "01", "fev": "02", "mar": "03", "abr": "04", "mai": "05", "jun": "06",
    "jul": "07", "ago": "08", "set": "09", "out": "10", "nov": "11", "dez": "12"
}

# Função para padronizar datas
def padronizar_data(data):
    if not data or str(data).strip().lower() == "none":
        return None  # Ignora valores nulos ou vazios

    data = data.strip().lower()

    # Substitui meses abreviados em português por números correspondentes
    for mes, num in meses_abreviados.items():
        data = re.sub(rf"(\d{{1,2}})[-/\s]{mes}[-/\s](\d{{2,4}})", rf"\1/{num}/\2", data)

    # Formatos conhecidos para conversão direta
    formatos = [
        "%d/%m/%Y", "%d-%m-%Y", "%d/%m/%y", "%d-%m-%y",
        "%d/%m/%Y %H:%M", "%d/%m/%Y %H:%M:%S"
    ]

    for formato in formatos:
        try:
            return datetime.strptime(data, formato).strftime("%d/%m/%Y")
        except ValueError:
            continue

    try:
        return parser.parse(data, dayfirst=True).strftime("%d/%m/%Y")
    except Exception:
        return data  # Retorna o valor original se não conseguir converter

# Lista de colunas padrão da tabela "escolas_full"
colunas_escolas_full = [
    "DRE", "CODESC", "TIPOESC", "NOMESC", "NOMESCOF", "CEU", "DIRETORIA", "SUBPREF",
    "ENDERECO", "NUMERO", "BAIRRO", "CEP", "TEL1", "TEL2", "FAX", "SITUACAO", "CODDIST",
    "DISTRITO", "SETOR", "CODINEP", "CODCIE", "EH", "FX_ETARIA", "DT_CRIACAO", "ATO_CRIACAO",
    "DOM_CRIACAO", "DT_INI_FUNC", "DT_INI_CONV", "DT_AUTORIZA", "DT_EXTINCAO", "NOME_ANT",
    "REDE", "LATITUDE", "LONGITUDE", "DATABASE"
]

# Função para inserir os dados na tabela "escolas_full"
def inserir_dados_escolas_full():
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()

    # Obtém todas as tabelas do banco
    cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
    tabelas = cursor.fetchall()

    # Filtra apenas tabelas relacionadas a "escolas"
    tabelas_escolas = [
        t[0] for t in tabelas if 'escola' in t[0].lower() 
        and 'dicionario' not in t[0].lower() 
        and 'full' not in t[0].lower()
    ]

    for tabela in tabelas_escolas:
        # Obtém informações das colunas da tabela
        cursor.execute(f'PRAGMA table_info("{tabela}");')
        colunas = cursor.fetchall()
        nomes_colunas = [col[1] for col in colunas]

        # Filtra apenas colunas que existem na tabela "escolas_full"
        colunas_validas = [coluna for coluna in nomes_colunas if coluna in colunas_escolas_full]

        if not colunas_validas:
            print(f"⚠ Nenhuma coluna válida encontrada na tabela '{tabela}'. Pulando...")
            continue

        # Seleciona os dados da tabela atual
        query_select = f'SELECT "{'", "'.join(colunas_validas)}" FROM "{tabela}";'
        cursor.execute(query_select)
        dados = cursor.fetchall()

        # Padroniza os dados antes de inseri-los
        dados_padronizados = []
        for linha in dados:
            linha_convertida = list(linha)
            for i, coluna in enumerate(colunas_validas):
                if "DT_" in coluna or coluna == "DATABASE":
                    linha_convertida[i] = padronizar_data(str(linha_convertida[i]))
            dados_padronizados.append(tuple(linha_convertida))

        # Inserir os dados padronizados na tabela "escolas_full"
        query_insert = f"""
            INSERT INTO escolas_full ("{'", "'.join(colunas_validas)}") 
            VALUES ({', '.join(['?' for _ in colunas_validas])});
        """

        cursor.executemany(query_insert, dados_padronizados)
        print(f"✅ Inseridos dados da tabela '{tabela}' na tabela 'escolas_full' com datas padronizadas.")

    conn.commit()
    conn.close()

# Inserir os dados na tabela "escolas_full"
inserir_dados_escolas_full()

✅ Inseridos dados da tabela 'escolas-dez-2010' na tabela 'escolas_full' com datas padronizadas.
✅ Inseridos dados da tabela 'escolas-dez-2011' na tabela 'escolas_full' com datas padronizadas.
✅ Inseridos dados da tabela 'escolas-dez-2012' na tabela 'escolas_full' com datas padronizadas.
✅ Inseridos dados da tabela 'escolas-dez-2013' na tabela 'escolas_full' com datas padronizadas.
✅ Inseridos dados da tabela 'escolas-dez-2014' na tabela 'escolas_full' com datas padronizadas.
✅ Inseridos dados da tabela 'escolas-dez-2015' na tabela 'escolas_full' com datas padronizadas.
✅ Inseridos dados da tabela 'escolas122018' na tabela 'escolas_full' com datas padronizadas.
✅ Inseridos dados da tabela 'escolas122019' na tabela 'escolas_full' com datas padronizadas.
✅ Inseridos dados da tabela 'escolas122020' na tabela 'escolas_full' com datas padronizadas.
✅ Inseridos dados da tabela 'escolas122021' na tabela 'escolas_full' com datas padronizadas.
✅ Inseridos dados da tabela 'escolas122022' na tabel

# Criar Coluna ANO na tabela "escolas_full"

In [8]:
import sqlite3

# Caminho do banco de dados
db_path = r"H:\Projeto USE\data_analytics\db\test_analytics.db"

# Conectar ao banco de dados
conn = sqlite3.connect(db_path)
cursor = conn.cursor()

# Verificar se a coluna ANO já existe
cursor.execute("PRAGMA table_info(escolas_full);")
colunas_existentes = [col[1] for col in cursor.fetchall()]

if "ANO" not in colunas_existentes:
    # Adicionar a coluna ANO
    cursor.execute("ALTER TABLE escolas_full ADD COLUMN ANO INTEGER;")
    print("Coluna ANO adicionada com sucesso.")

# Atualizar a coluna ANO extraindo os últimos 4 caracteres de DATABASE (o ano)
cursor.execute("UPDATE escolas_full SET ANO = substr(DATABASE, -4, 4);")
print("Coluna ANO preenchida com os anos extraídos de DATABASE.")

# Salvar e fechar conexão
conn.commit()
conn.close()


Coluna ANO adicionada com sucesso.
Coluna ANO preenchida com os anos extraídos de DATABASE.


# Diagnosticar as colunas das tabelas que contém "idades" especificadas¶

In [13]:
import sqlite3

# Defina o caminho do seu banco de dados
db_path = r"H:\Projeto USE\data_analytics\db\test_analytics.db"

# Padrão de cabeçalhos
padrão_cabeçalhos = [
    "DRE", "CODESC", "TIPOESC", "NOMESC", "DISTRITO", "SETOR", "ANO", 
    "REDE", "MODAL", "DESCSERIE", "PERIODO", "TURNO", "DESCTURNO", 
    "SEXO", "IDADE", "NEE", "RACA", "QTDE", "DATABASE"
]

# Função para verificar os cabeçalhos das tabelas
def verificar_cabecalhos():
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()

    # Pegue a lista de tabelas
    cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
    tabelas = cursor.fetchall()

    # Filtrar tabelas que contém 'idades' no nome
    tabelas_com_idades = [
        t[0] for t in tabelas 
        if 'idades' in t[0].lower()  # Tabelas com 'idades' no nome
    ]

    # Verificar cabeçalhos de cada tabela
    tabelas_com_problema = {}
    for tabela in tabelas_com_idades:
        # Envolver o nome da tabela com aspas invertidas para evitar problemas com caracteres especiais
        cursor.execute(f"PRAGMA table_info(`{tabela}`);")
        colunas = cursor.fetchall()
        nomes_colunas = [col[1] for col in colunas]
        
        # Comparar com o padrão
        colunas_faltando = list(set(padrão_cabeçalhos) - set(nomes_colunas))
        colunas_diferentes = list(set(nomes_colunas) - set(padrão_cabeçalhos))

        if colunas_faltando or colunas_diferentes:
            tabelas_com_problema[tabela] = {
                "colunas_faltando": colunas_faltando,
                "colunas_diferentes": colunas_diferentes
            }

    # Fechar conexão
    conn.close()
    
    return tabelas_com_problema

# Executar e exibir resultados
problemas = verificar_cabecalhos()
if problemas:
    for tabela, info in problemas.items():
        print(f"Tabela: {tabela}")
        
        if info["colunas_faltando"]:
            print("Colunas faltando:")
            for coluna in info["colunas_faltando"]:
                print(f"  - {coluna}")
        
        if info["colunas_diferentes"]:
            print("Colunas com nome diferente:")
            for coluna in info["colunas_diferentes"]:
                print(f"  - {coluna}")
        
        print()
else:
    print("Todas as tabelas com 'idades' no nome estão com os cabeçalhos padrão.")


Tabela: idadeserieneeraca-r33
Colunas com nome diferente:
  - SERIEV
  - MODALID

Tabela: idadeserieneeracadez17
Colunas com nome diferente:
  - SERIEV
  - MODALID



# Verificar se a coluna 'SETEDU' existe e renomear para 'SETOR'

In [20]:
import sqlite3

# Caminho do banco de dados
db_path = r"H:\Projeto USE\data_analytics\db\test_analytics.db"

# Função para corrigir SETEDU para SETOR
def corrigir_setedu_para_setor(tabela):
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()

    # Obter as colunas da tabela
    cursor.execute(f"PRAGMA table_info(`{tabela}`);")
    colunas = [col[1] for col in cursor.fetchall()]

    # Verificar se a coluna 'SETEDU' existe e renomear para 'SETOR'
    if "SETEDU" in colunas:
        if "SETOR" not in colunas:
            try:
                cursor.execute(f'ALTER TABLE `{tabela}` RENAME COLUMN `SETEDU` TO `SETOR`;')
                print(f"✅ Corrigido 'SETEDU' para 'SETOR' na tabela '{tabela}'")
            except sqlite3.OperationalError as e:
                print(f"⚠️ Erro ao renomear 'SETEDU' para 'SETOR' na tabela '{tabela}': {e}")
        else:
            print(f"⚠️ A coluna 'SETOR' já existe na tabela '{tabela}', 'SETEDU' não pode ser renomeada.")
    else:
        print(f"ℹ️ A coluna 'SETEDU' não existe na tabela '{tabela}'.")

    conn.commit()
    conn.close()

# Listar as tabelas que precisam da correção
tabelas_para_corrigir = ["idadeserieneeraca-r33", "idadeserieneeracadez17"]

# Aplicar a correção para cada tabela
for tabela in tabelas_para_corrigir:
    corrigir_setedu_para_setor(tabela)


ℹ️ A coluna 'SETEDU' não existe na tabela 'idadeserieneeraca-r33'.
ℹ️ A coluna 'SETEDU' não existe na tabela 'idadeserieneeracadez17'.


# Ajustar as tabelas "idades"

In [4]:
import sqlite3
import difflib
import re

# Função para limpar os nomes das colunas, transformando tudo em minúsculas e removendo caracteres especiais
def limpar_nome_coluna(nome):
    nome = nome.lower()  # Transformar para minúsculas
    nome = re.sub(r'\W+', '', nome)  # Remover caracteres especiais (não alfanuméricos)
    return nome

# Função para corrigir ou adicionar as colunas
def corrigir_ou_adicionar_colunas(tabela, colunas_faltando, colunas_diferentes):
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()

    # Limpar os nomes das colunas faltantes e com nome diferente
    colunas_faltando_limpa = [limpar_nome_coluna(coluna) for coluna in colunas_faltando]
    colunas_diferentes_limpa = [limpar_nome_coluna(coluna) for coluna in colunas_diferentes]

    for coluna_faltando, coluna_faltando_limpa in zip(colunas_faltando, colunas_faltando_limpa):
        # Encontrar coluna com nome semelhante (por exemplo, uma letra a mais ou minúscula/maiúscula)
        colunas_similares = difflib.get_close_matches(coluna_faltando_limpa, colunas_diferentes_limpa, n=1, cutoff=0.8)

        if colunas_similares:
            coluna_errada_limpa = colunas_similares[0]
            coluna_errada = colunas_diferentes[colunas_diferentes_limpa.index(coluna_errada_limpa)]
            # Corrigir o nome da coluna
            cursor.execute(f"ALTER TABLE `{tabela}` RENAME COLUMN `{coluna_errada}` TO `{coluna_faltando}`;")
            print(f"Corrigido nome da coluna '{coluna_errada}' para '{coluna_faltando}' na tabela '{tabela}'")
        else:
            # Verificar se a coluna já existe antes de adicionar
            cursor.execute(f"PRAGMA table_info(`{tabela}`);")
            colunas_existentes = [col[1] for col in cursor.fetchall()]
            
            if coluna_faltando not in colunas_existentes:
                # Se não encontrar, criar a coluna faltante com dados nulos
                cursor.execute(f"ALTER TABLE `{tabela}` ADD COLUMN `{coluna_faltando}` TEXT;")
                print(f"Criada a coluna faltante '{coluna_faltando}' na tabela '{tabela}'")
            else:
                print(f"A coluna '{coluna_faltando}' já existe na tabela '{tabela}'")

    conn.commit()
    conn.close()

# Função para verificar os cabeçalhos e corrigir
def verificar_e_corrigir_cabecalhos():
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()

    # Pegue a lista de tabelas
    cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
    tabelas = cursor.fetchall()

    # Filtrar tabelas que contém 'escola' no nome e não contém 'dicionario'
    tabelas_escolas = [t[0] for t in tabelas if 'idades' in t[0].lower() and 'dicionario' not in t[0].lower()]

    # Verificar cabeçalhos de cada tabela
    for tabela in tabelas_escolas:
        # Envolver o nome da tabela com aspas invertidas para evitar problemas com caracteres especiais
        cursor.execute(f"PRAGMA table_info(`{tabela}`);")
        colunas = cursor.fetchall()
        nomes_colunas = [col[1] for col in colunas]

        # Comparar com o padrão
        colunas_faltando = list(set(padrão_cabeçalhos) - set(nomes_colunas))
        colunas_diferentes = list(set(nomes_colunas) - set(padrão_cabeçalhos))

        if colunas_faltando or colunas_diferentes:
            print(f"Tabela: {tabela}")
            
            # Corrigir ou adicionar as colunas faltantes e com nome diferente
            corrigir_ou_adicionar_colunas(tabela, colunas_faltando, colunas_diferentes)
    
    conn.close()

# Executar e corrigir cabeçalhos
verificar_e_corrigir_cabecalhos()


Tabela: idadeserieneeraca-r33
Tabela: idadeserieneeracadez17
Tabela: idadeserieneeracadez18
Corrigido nome da coluna 'qtd' para 'QTDE' na tabela 'idadeserieneeracadez18'
Corrigido nome da coluna 'tipoesc' para 'TIPOESC' na tabela 'idadeserieneeracadez18'
Corrigido nome da coluna 'modal' para 'MODAL' na tabela 'idadeserieneeracadez18'
Corrigido nome da coluna 'idade' para 'IDADE' na tabela 'idadeserieneeracadez18'
Corrigido nome da coluna 'descserie' para 'DESCSERIE' na tabela 'idadeserieneeracadez18'
Corrigido nome da coluna 'nomesc' para 'NOMESC' na tabela 'idadeserieneeracadez18'
Corrigido nome da coluna 'codes' para 'CODESC' na tabela 'idadeserieneeracadez18'
Corrigido nome da coluna 'nee' para 'NEE' na tabela 'idadeserieneeracadez18'
Corrigido nome da coluna 'database' para 'DATABASE' na tabela 'idadeserieneeracadez18'
Corrigido nome da coluna 'descturno' para 'DESCTURNO' na tabela 'idadeserieneeracadez18'
Corrigido nome da coluna 'ano' para 'ANO' na tabela 'idadeserieneeracadez18'

# Criar um nova tabela "alunos_full"

In [9]:
import sqlite3

# Caminho do banco de dados
db_path = r"H:\Projeto USE\data_analytics\db\test_analytics.db"

# Função para criar a tabela "alunos_full"
def criar_tabela_alunos_full():
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()

    # Criar a tabela "alunos_full"
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS alunos_full (
            DRE TEXT,
            CODESC BIGINT,
            TIPOESC TEXT,
            NOMESC TEXT,
            DISTRITO TEXT,
            SETOR TEXT,
            ANO BIGINT,
            REDE TEXT,
            MODAL TEXT,
            DESCSERIE TEXT,
            PERIODO TEXT,
            TURNO BIGINT,
            DESCTURNO TEXT,
            SEXO TEXT,
            IDADE BIGINT,
            NEE TEXT,
            RACA TEXT,
            QTDE BIGINT,
            DATABASE TEXT
        );
    """)

    conn.commit()
    conn.close()

# Criar a tabela "alunos_full"
criar_tabela_alunos_full()


# Inserir os dados na nova tabela criada "alunos_full"

In [10]:
import sqlite3
from datetime import datetime
from dateutil import parser
import re

# Caminho do banco de dados
db_path = r"H:\Projeto USE\data_analytics\db\test_analytics.db"

# Mapeamento de meses abreviados em português para números
meses_abreviados = {
    "jan": "01", "fev": "02", "mar": "03", "abr": "04", "mai": "05", "jun": "06",
    "jul": "07", "ago": "08", "set": "09", "out": "10", "nov": "11", "dez": "12"
}

# Função para padronizar datas
def padronizar_data(data):
    if not data or str(data).strip().lower() == "none":
        return None  # Ignora valores nulos ou vazios

    data = data.strip().lower()

    # Substitui meses abreviados em português por números correspondentes
    for mes, num in meses_abreviados.items():
        data = re.sub(rf"(\d{{1,2}})[-/\s]{mes}[-/\s](\d{{2,4}})", rf"\1/{num}/\2", data)

    # Formatos conhecidos para conversão direta
    formatos = [
        "%d/%m/%Y", "%d-%m-%Y", "%d/%m/%y", "%d-%m-%y",
        "%d/%m/%Y %H:%M", "%d/%m/%Y %H:%M:%S"
    ]

    for formato in formatos:
        try:
            return datetime.strptime(data, formato).strftime("%d/%m/%Y")
        except ValueError:
            continue

    try:
        return parser.parse(data, dayfirst=True).strftime("%d/%m/%Y")
    except Exception:
        return data  # Retorna o valor original se não conseguir converter

# Lista de colunas padrão para a tabela "alunos_full"
colunas_alunos_full = [
    "DRE", "CODESC", "TIPOESC", "NOMESC", "DISTRITO", "SETOR", "ANO", "REDE",
    "MODAL", "DESCSERIE", "PERIODO", "TURNO", "DESCTURNO", "SEXO", "IDADE",
    "NEE", "RACA", "QTDE", "DATABASE"
]

# Função para inserir os dados na tabela "alunos_full"
def inserir_dados_alunos_full():
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()

    # Selecionar todas as tabelas que contêm "idades" no nome, mas não "dicionario" nem "full"
    cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
    tabelas = cursor.fetchall()

    tabelas_idades = [t[0] for t in tabelas if 'idades' in t[0].lower() and 'dicionario' not in t[0].lower() and 'full' not in t[0].lower()]

    for tabela in tabelas_idades:
        cursor.execute(f"PRAGMA table_info(`{tabela}`);")
        colunas = cursor.fetchall()
        nomes_colunas = [col[1] for col in colunas]

        # Filtrar apenas as colunas que fazem parte da tabela "alunos_full"
        colunas_validas = [coluna for coluna in nomes_colunas if coluna in colunas_alunos_full]

        if not colunas_validas:  
            print(f"⚠ Nenhuma coluna válida encontrada na tabela '{tabela}'. Pulando...")
            continue

        # Selecionar os dados da tabela atual
        query_select = f'SELECT "{'", "'.join(colunas_validas)}" FROM "{tabela}";'
        cursor.execute(query_select)
        dados = cursor.fetchall()

        # Padronizar as datas antes de inseri-las
        dados_padronizados = []
        for linha in dados:
            linha_convertida = list(linha)
            for i, coluna in enumerate(colunas_validas):
                if coluna == "DATABASE":
                    linha_convertida[i] = padronizar_data(str(linha_convertida[i]))
            dados_padronizados.append(tuple(linha_convertida))

        # Inserir os dados padronizados na tabela "alunos_full"
        query_insert = f"""
            INSERT INTO alunos_full ("{'", "'.join(colunas_validas)}") 
            VALUES ({', '.join(['?' for _ in colunas_validas])});
        """
        cursor.executemany(query_insert, dados_padronizados)

        print(f"✅ Inseridos dados da tabela '{tabela}' na tabela 'alunos_full' com datas padronizadas.")

    conn.commit()
    conn.close()

# Inserir os dados na tabela "alunos_full"
inserir_dados_alunos_full()

✅ Inseridos dados da tabela 'idadeserieneeraca-r33' na tabela 'alunos_full' com datas padronizadas.
✅ Inseridos dados da tabela 'idadeserieneeracadez17' na tabela 'alunos_full' com datas padronizadas.
✅ Inseridos dados da tabela 'idadeserieneeracadez18' na tabela 'alunos_full' com datas padronizadas.
✅ Inseridos dados da tabela 'idadeserieneeracadez19' na tabela 'alunos_full' com datas padronizadas.
✅ Inseridos dados da tabela 'idadeserieneeracadez20' na tabela 'alunos_full' com datas padronizadas.
✅ Inseridos dados da tabela 'idadeserieneeracadez21' na tabela 'alunos_full' com datas padronizadas.
✅ Inseridos dados da tabela 'idadeserieneeracadez22' na tabela 'alunos_full' com datas padronizadas.
✅ Inseridos dados da tabela 'idadeserieneeracadez23' na tabela 'alunos_full' com datas padronizadas.


# Inspeção inicial dos dados

In [97]:
import sqlite3
import pandas as pd

# Conectar ao banco
db_path = r"H:\Projeto USE\data_analytics\db\test_analytics.db"
conn = sqlite3.connect(db_path)

# Listar tabelas disponíveis
query = "SELECT name FROM sqlite_master WHERE type='table';"
tabelas = pd.read_sql(query, conn)
print("Tabelas disponíveis:\n", tabelas)


Tabelas disponíveis:
                        name
0         dicionarioescolas
1          escolas-dez-2010
2          escolas-dez-2011
3          escolas-dez-2012
4          escolas-dez-2013
5          escolas-dez-2014
6          escolas-dez-2015
7             escolas122018
8             escolas122019
9             escolas122020
10            escolas122021
11            escolas122022
12            escolas122023
13               escolasr34
14        escolasr34dez2017
15  dicionariopefileducando
16    idadeserieneeraca-r33
17   idadeserieneeracadez17
18   idadeserieneeracadez18
19   idadeserieneeracadez19
20   idadeserieneeracadez20
21   idadeserieneeracadez21
22   idadeserieneeracadez22
23   idadeserieneeracadez23
24              alunos_full
25             escolas_full
26           escolas_alunos


# Ver algumas amostras das tabelas principais

In [42]:
df_escolas = pd.read_sql("SELECT * FROM escolas_full LIMIT 5;", conn)
df_alunos = pd.read_sql("SELECT * FROM alunos_full LIMIT 5;", conn)

print("Escolas:")
display(df_escolas)

print("Alunos:")
display(df_alunos)


Escolas:


,DRE,CODESC,TIPOESC,NOMESC,NOMESCOF,CEU,DIRETORIA,SUBPREF,ENDERECO,NUMERO,...,DT_INI_FUNC,DT_INI_CONV,DT_AUTORIZA,DT_EXTINCAO,NOME_ANT,REDE,LATITUDE,LONGITUDE,DATABASE,ANO
0,BT,307207,ESC.PART.,A HEBRAICA,A HEBRAICA,None,BUTANTA,PINHEIROS,HUNGRIA,1000,...,None,None,06/02/2010,None,None,PAR,-23578930,-46694078,31/12/2010,2010
1,BT,307412,ESC.PART.,ABCD SOFIA,ABCD SOFIA,None,BUTANTA,BUTANTA,OTÁVIO PEDREIRO ROSA,212,...,None,None,12/11/2010,None,None,PAR,-23583388,-46760545,31/12/2010,2010
2,BT,307131,ESC.PART.,"ACONCHEGO, ESCOLA","ACONCHEGO, ESCOLA",None,BUTANTA,BUTANTA,QUITANDUBA,50,...,None,None,None,None,None,PAR,-23583061,-46715523,31/12/2010,2010
3,BT,306549,ESC.PART.,ADOLPHE FERRIERE,ADOLPHE FERRIERE,None,BUTANTA,BUTANTA,RUA FRANCISCO PRETO,457,...,None,None,21/10/2003,None,None,PAR,-23603215,-46726799,31/12/2010,2010
4,BT,92754,EMEF,"ALCIDES GONCALVES ETCHEGOYEN, GEN.","ALCIDES GONCALVES ETCHEGOYEN, GEN.",None,BUTANTA,BUTANTA,RUA ADHERBAL STRESSER,686,...,None,None,10/04/1981,None,ARPOADOR,DIR,-23594177,-46791273,31/12/2010,2010


Alunos:


,DRE,CODESC,TIPOESC,NOMESC,DISTRITO,SETOR,ANO,REDE,MODAL,DESCSERIE,PERIODO,TURNO,DESCTURNO,SEXO,IDADE,NEE,RACA,QTDE,DATABASE
0,BT,191,EMEF,"ALIPIO CORREA NETO, PROF.",VILA SONIA,9404,2016,DIR,ATCOMP,JOGOS EDUCATIVO,DIURNO,3,Tarde,F,12,NAO POSSUI,PARDA,2,30/09/2016
1,BT,191,EMEF,"ALIPIO CORREA NETO, PROF.",VILA SONIA,9404,2016,DIR,ATCOMP,JOGOS EDUCATIVO,DIURNO,3,Tarde,F,13,NAO POSSUI,NÃO DECLARADA,1,30/09/2016
2,BT,191,EMEF,"ALIPIO CORREA NETO, PROF.",VILA SONIA,9404,2016,DIR,ATCOMP,JOGOS EDUCATIVO,DIURNO,3,Tarde,F,15,NAO POSSUI,PARDA,1,30/09/2016
3,BT,191,EMEF,"ALIPIO CORREA NETO, PROF.",VILA SONIA,9404,2016,DIR,ATCOMP,JOGOS EDUCATIVO,DIURNO,3,Tarde,M,11,NAO POSSUI,NÃO DECLARADA,1,30/09/2016
4,BT,191,EMEF,"ALIPIO CORREA NETO, PROF.",VILA SONIA,9404,2016,DIR,ATCOMP,JOGOS EDUCATIVO,DIURNO,3,Tarde,M,11,NAO POSSUI,PARDA,1,30/09/2016


# Valores Ausentes

In [117]:
print("Valores ausentes em escolas_full:")
print(df_escolas.isnull().sum())

print("\nValores ausentes em alunos_full:")
print(df_alunos.isnull().sum())


Valores ausentes em escolas_full:
IDADE     0
CODESC    0
dtype: int64

Valores ausentes em alunos_full:
DRE                  0
CODESC               0
TIPOESC              0
NOMESC               0
DISTRITO             0
SETOR                0
ANO                  0
REDE              9473
MODAL            22424
DESCSERIE            6
PERIODO              0
TURNO                0
DESCTURNO            0
SEXO               521
IDADE              503
NEE                  0
RACA            178666
QTDE                 0
DATABASE             0
PERFIL_IDADE         0
PERFIL             521
dtype: int64


# Qual a proporção entre alunos pela perspectiva dos perfis.

In [77]:
import sqlite3
import pandas as pd

# Conectar ao banco de dados
db_path = r"H:\Projeto USE\data_analytics\db\test_analytics.db"
conn = sqlite3.connect(db_path)

# Carregar os dados da tabela de alunos
df_alunos = pd.read_sql("SELECT * FROM alunos_full;", conn)

# Fechar conexão
conn.close()

# Calcular a distribuição de alunos por SEXO
df_sexo = df_alunos.groupby("SEXO", as_index=False)["QTDE"].sum()

# Calcular o percentual de cada SEXO
total_alunos = df_sexo["QTDE"].sum()
df_sexo["PERCENTUAL"] = (df_sexo["QTDE"] / total_alunos) * 100

# Exibir as tabelas
print("\nDistribuição de alunos por SEXO (com percentuais):\n")
print(df_sexo.to_string(index=False))



Distribuição de alunos por SEXO (com percentuais):

SEXO    QTDE  PERCENTUAL
   F 5014251   49.495979
   M 5116372   50.504021


In [102]:
# Calcular a proporção de alunos por IDADE
df_idade['PERCENTUAL'] = (df_idade['QTDE'] / df_idade['QTDE'].sum()) * 100

# Ordenar por QTDE em ordem decrescente
df_idade_sorted = df_idade.sort_values(by='QTDE', ascending=False)

# Exibir a tabela com a proporção e o percentual ordenada
print("\nProporção de alunos por QTDE (em ordem decrescente):")
print(df_idade_sorted[['IDADE', 'QTDE', 'PERCENTUAL']].to_string(index=False))



Proporção de alunos por QTDE (em ordem decrescente):
 IDADE   QTDE  PERCENTUAL
   3.0 223422    7.491284
   2.0 218859    7.338288
   4.0 207481    6.956786
   1.0 168304    5.643191
   5.0 166999    5.599435
  12.0 165860    5.561244
  13.0 163518    5.482718
  11.0 163220    5.472726
  14.0 154576    5.182895
  10.0 149418    5.009948
   6.0 141582    4.747209
   9.0 130217    4.366144
   8.0 111440    3.736555
  15.0 109756    3.680091
   7.0  92444    3.099624
  16.0  60994    2.045114
   0.0  48341    1.620862
  17.0  34630    1.161135
  18.0  21716    0.728132
  19.0  15193    0.509418
  20.0  11682    0.391695
  47.0  10113    0.339086
  48.0  10094    0.338449
  45.0  10026    0.336169
  43.0   9975    0.334459
  44.0   9962    0.334023
  46.0   9950    0.333621
  49.0   9861    0.330637
  42.0   9828    0.329530
  41.0   9757    0.327150
  50.0   9618    0.322489
  21.0   9583    0.321316
  40.0   9551    0.320243
  52.0   9527    0.319438
  51.0   9508    0.318801
  39.0   9

In [84]:
# Agrupar pelos perfis de SEXO e IDADE e calcular a quantidade total
df_perfil = df_alunos.groupby(["SEXO", "IDADE"], as_index=False)["QTDE"].sum()

# Calcular o percentual do total geral
total_geral = df_perfil['QTDE'].sum()  # Calcular o total geral de alunos
df_perfil['PERCENTUAL'] = (df_perfil['QTDE'] / total_geral) * 100

# Ordenar os resultados por percentual de forma decrescente
df_perfil_sorted = df_perfil.sort_values(by='PERCENTUAL', ascending=False)

# Exibir a tabela com o perfil, quantidade total e percentual
print("\nPerfil de alunos (SEXO + IDADE), quantidade total e percentual:")
print(df_perfil_sorted.to_string(index=False))



Perfil de alunos (SEXO + IDADE), quantidade total e percentual:
SEXO  IDADE   QTDE  PERCENTUAL
   M    5.0 498102    4.916795
   M    4.0 490662    4.843355
   F    5.0 473595    4.674885
   F    4.0 463166    4.571940
   M    3.0 460403    4.544666
   F    3.0 429971    4.244270
   M    2.0 400676    3.955097
   M    6.0 381931    3.770064
   F    2.0 369965    3.651947
   F    6.0 364382    3.596837
   M   12.0 292432    2.886614
   M   13.0 289678    2.859429
   M   11.0 282843    2.791961
   M   14.0 279853    2.762446
   F   12.0 270525    2.670369
   M    1.0 269428    2.659540
   M   10.0 269172    2.657013
   F   11.0 266582    2.631447
   F   13.0 263242    2.598478
   F   14.0 254889    2.516025
   F   10.0 253260    2.499945
   M    9.0 250906    2.476708
   F    1.0 244604    2.414501
   F    9.0 235766    2.327261
   M    8.0 227216    2.242863
   F    8.0 215946    2.131616
   M    7.0 208661    2.059706
   F    7.0 199244    1.966750
   M   15.0 173009    1.707782
   F 

# Definição de perfis a partir da idade e gênero

In [86]:
import pandas as pd

# Função para criar o label de perfil por faixa etária
def label_perfil_idade(idade):
    if idade < 2:
        return "Abaixo de 2 anos"
    elif 3 <= idade <= 18:
        return f"{idade//2*2}-{(idade//2)*2+1} anos"
    else:
        return "Adultos"

# Aplicar a função para criar a coluna de PERFIL_IDADE
df_alunos['PERFIL_IDADE'] = df_alunos['IDADE'].apply(label_perfil_idade)

# Criar a coluna de PERFIL combinando SEXO e PERFIL_IDADE
df_alunos['PERFIL'] = df_alunos['SEXO'] + " " + df_alunos['PERFIL_IDADE']

# Agrupar pelos perfis de SEXO e PERFIL_IDADE e calcular a quantidade total
df_perfil = df_alunos.groupby(["SEXO", "PERFIL"], as_index=False)["QTDE"].sum()

# Calcular o percentual do total geral
total_geral = df_perfil['QTDE'].sum()  # Calcular o total geral de alunos
df_perfil['PERCENTUAL'] = (df_perfil['QTDE'] / total_geral) * 100

# Ordenar os resultados por percentual de forma decrescente
df_perfil_sorted = df_perfil.sort_values(by='PERCENTUAL', ascending=False)

# Exibir a tabela com o perfil, quantidade total e percentual
print("\nPerfil de alunos (SEXO + PERFIL), quantidade total e percentual:")
print(df_perfil_sorted.to_string(index=False))



Perfil de alunos (SEXO + PERFIL), quantidade total e percentual:
SEXO             PERFIL   QTDE  PERCENTUAL
   M     M 4.0-5.0 anos 988764    9.760150
   F     F 4.0-5.0 anos 936761    9.246825
   F          F Adultos 798680    7.883819
   M     M 6.0-7.0 anos 590592    5.829770
   M   M 12.0-13.0 anos 582110    5.746043
   M          M Adultos 563871    5.566005
   F     F 6.0-7.0 anos 563626    5.563587
   M   M 10.0-11.0 anos 552015    5.448974
   F   F 12.0-13.0 anos 533767    5.268847
   F   F 10.0-11.0 anos 519842    5.131392
   M     M 8.0-9.0 anos 478122    4.719572
   M     M 2.0-3.0 anos 460403    4.544666
   M   M 14.0-15.0 anos 452862    4.470229
   F     F 8.0-9.0 anos 451712    4.458877
   F     F 2.0-3.0 anos 429971    4.244270
   F   F 14.0-15.0 anos 397762    3.926333
   M M Abaixo de 2 anos 327183    3.229643
   F F Abaixo de 2 anos 296835    2.930076
   M   M 16.0-17.0 anos 100619    0.993216
   F   F 16.0-17.0 anos  69295    0.684015
   M   M 18.0-19.0 anos  19831 

# Qual a densidade entre perfis e escola.

In [121]:
import sqlite3
import pandas as pd

# Função para conectar ao banco de dados
def conectar_db(db_path):
    try:
        conn = sqlite3.connect(db_path)
        return conn
    except sqlite3.Error as e:
        print(f"Erro ao conectar ao banco de dados: {e}")
        return None

# Função para carregar os dados da tabela
def carregar_dados(conn, query):
    try:
        df = pd.read_sql(query, conn)
        return df
    except Exception as e:
        print(f"Erro ao carregar dados: {e}")
        return pd.DataFrame()

# Função para criar o label de perfil por faixa etária
def label_perfil_idade(idade):
    if idade < 2:
        return "Abaixo de 2 anos"
    elif 3 <= idade <= 18:
        return f"{(idade // 2) * 2}-{(idade // 2) * 2 + 1} anos"
    else:
        return "Adultos"

# Função principal para processar os dados
def processar_dados(df):
    # Verificar valores nulos nas colunas chave
    if df[['IDADE', 'SEXO', 'QTDE']].isnull().any().any():
        print("Existem valores nulos nas colunas 'IDADE', 'SEXO' ou 'QTDE'.")
        print(df[['IDADE', 'SEXO', 'QTDE']].isnull().sum())
        return pd.DataFrame()

    # Verificar o conteúdo das colunas 'SEXO' e 'IDADE'
    print("Valores únicos de SEXO:", df['SEXO'].unique())
    print("Valores únicos de IDADE:", df['IDADE'].unique())

    # Aplicar a função para criar a coluna de PERFIL_IDADE
    df['PERFIL_IDADE'] = df['IDADE'].apply(label_perfil_idade)

    # Verificar se a coluna PERFIL_IDADE foi criada corretamente
    print("Valores únicos de PERFIL_IDADE:", df['PERFIL_IDADE'].unique())

    # Criar a coluna de PERFIL combinando SEXO e PERFIL_IDADE
    df['PERFIL'] = df['SEXO'] + " " + df['PERFIL_IDADE']

    # Verificar os perfis criados
    print("Perfis únicos:", df['PERFIL'].unique())

    # Agrupar pelos perfis de SEXO e PERFIL_IDADE por escola (NOMESCOF) e calcular a quantidade total de alunos
    df_perfil_escola = df.groupby(["NOMESCOF", "SEXO", "PERFIL", "ANO"], as_index=False)["QTDE"].sum()

    # Criar uma tabela pivot para que cada perfil de alunos seja uma coluna
    df_pivot = df_perfil_escola.pivot_table(index="NOMESCOF", columns="PERFIL", values="QTDE", aggfunc="sum", fill_value=0)

    # Verificar a tabela pivot antes de calcular a densidade
    print("Tabela Pivot antes da densidade:")
    print(df_pivot.head())

    # Calcular a densidade de cada perfil por escola
    df_pivot_density = df_pivot.div(df_pivot.sum(axis=1), axis=0) * 100

    return df_pivot_density

# Função para exibir a tabela com a densidade
def exibir_densidade(df_pivot_density):
    if df_pivot_density.empty:
        print("A tabela de densidade está vazia.")
    else:
        print("\nDensidade dos perfis por escola:")
        print(df_pivot_density.to_string(index=True))

# Caminho para o banco de dados
db_path = r"C:\Users\Pessoal\Desktop\Projeto USE\test_analytics.db"

# Conectar ao banco de dados
conn = conectar_db(db_path)

if conn:
    # Carregar os dados da tabela e filtrar os anos >= 2020, mas sem limitar a quantidade
    query = """
    SELECT * FROM escolas_alunos 
    WHERE ANO >= 2020;
    """
    df_escolas_alunos = carregar_dados(conn, query)

    # Verificar se os dados foram carregados corretamente
    if not df_escolas_alunos.empty:
        print("Dados carregados:")
        print(df_escolas_alunos.head())  # Exibir os primeiros dados

        # Processar os dados
        df_pivot_density = processar_dados(df_escolas_alunos)

        # Exibir a tabela com a densidade por perfil em cada escola
        exibir_densidade(df_pivot_density)
    else:
        print("Nenhum dado encontrado para os critérios especificados.")
    
    # Fechar a conexão
    conn.close()


Dados carregados:
  DRE    TIPOESC                               NOMESC  \
0  BT  CEI DIRET  ALOYSIO DE MENEZES GREENHALGH, VER.   
1  BT  CEI DIRET  ALOYSIO DE MENEZES GREENHALGH, VER.   
2  BT  CEI DIRET  ALOYSIO DE MENEZES GREENHALGH, VER.   
3  BT  CEI DIRET  ALOYSIO DE MENEZES GREENHALGH, VER.   
4  BT  CEI DIRET  ALOYSIO DE MENEZES GREENHALGH, VER.   

                              NOMESCOF   CEU DIRETORIA  SUBPREF  \
0  ALOYSIO DE MENEZES GREENHALGH, VER.  None   BUTANTA  BUTANTA   
1  ALOYSIO DE MENEZES GREENHALGH, VER.  None   BUTANTA  BUTANTA   
2  ALOYSIO DE MENEZES GREENHALGH, VER.  None   BUTANTA  BUTANTA   
3  ALOYSIO DE MENEZES GREENHALGH, VER.  None   BUTANTA  BUTANTA   
4  ALOYSIO DE MENEZES GREENHALGH, VER.  None   BUTANTA  BUTANTA   

                    ENDERECO NUMERO                BAIRRO  ...  LATITUDE  \
0  RUA DOUTOR FRANCISCO PATI    375  CIDADE SAO FRANCISCO  ... -23557659   
1  RUA DOUTOR FRANCISCO PATI    375  CIDADE SAO FRANCISCO  ... -23557659   
2  RUA D

In [ ]:
import sqlite3
import pandas as pd

# Conectar ao banco de dados
db_path = r"H:\Projeto USE\data_analytics\db\test_analytics.db"
conn = sqlite3.connect(db_path)

# Carregar os dados da tabela escolas_alunos
df_escolas_alunos = pd.read_sql("SELECT * FROM escolas_alunos;", conn)

# Fechar a conexão
conn.close()

# Calcular o total de alunos por DISTRITO (região de São Paulo)
df_distrito = df_escolas_alunos.groupby('DISTRITO', as_index=False)['QTDE'].sum()

# Calcular o total geral de alunos
total_alunos = df_escolas_alunos['QTDE'].sum()

# Calcular a proporção de alunos por DISTRITO
df_distrito['PERCENTUAL'] = (df_distrito['QTDE'] / total_alunos) * 100

# Exibir a tabela com a quantidade de alunos e a proporção por DISTRITO
print("\nProporção de alunos por DISTRITO (região de São Paulo):")
print(df_distrito.to_string(index=False))
